# 제출용 파일 2/3 · TabM 학습 (전처리 통일) — GPU/MPS/CPU 자동 감지

해커톤 제출 검증용. **`train.csv`에서 TabM(from-scratch)을 5-fold 학습**하고 OOF·test 예측을 저장합니다.
전처리(quantile·robust scaling)를 통일한 버전으로, TabM 단독 OOF AUC ≈ **0.74002** (최종 6멤버 블렌드의 최강 단일 멤버·가중치 0.341).
디바이스는 `cuda → mps → cpu` 순으로 자동 감지합니다.
(트리·선형·NN은 파일 1=케글, 최종 6멤버 블렌드·`submission_final.csv`는 파일 3. 최종 Public LB **0.7424857289**.)

**규정 준수 (test 누수 차단)**
- TabM 내부 전처리(quantile·robust scaling)는 **매 fold의 `.fit()` 데이터(=fold-train)에서만** 적합 → fold-내부·누수0.
- test는 어떤 fit에도 미투입. OCC/AGE 순서맵은 고정 사전(데이터 비의존). 외부데이터·유사라벨링 미사용.

**재현성 — from-scratch 딥의 한계와 방어**
- 시드·`random_state` 고정 + 결정성 시도(`use_deterministic_algorithms(True)`).
- 단 TabM/MLP의 일부 GPU·MPS 커널은 **결정적 구현이 없어**, 엄격 결정성 시 크래시 → **경고 후 비결정 모드로 폴백**. 즉 TabM 확률은 하드웨어(CUDA/MPS/드라이버) 의존으로 소수점 4째자리 흔들림 가능.
- 이 흔들림은 **파일 3의 랭크(rank) 블렌딩이 흡수** — 확률이 미세히 변해도 *순위*는 거의 불변이라 AUC가 보존됨.

**입력:** `train.csv` · `test.csv`   **환경:** GPU/MPS/CPU 어디서나(자동 감지)
**산출:** `oof_tabm.csv` · `test_tabm.csv` (파일 3 입력)   **라이브러리:** pytabkit(Apache-2.0)

In [27]:
import sys, subprocess, gc, time
import numpy as np
import pandas as pd
import torch

# 필수 라이브러리 설치
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytabkit"], check=False)
from pytabkit import TabM_D_Classifier

# 나머지 코드는 이전과 동일합니다
DEV = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("[tabm] device", DEV)

# [이하 v2v3 파생변수 생성기 및 나머지 함수들은 아까 드린 것과 동일하게 붙여넣으시면 됩니다]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

[tabm] device cuda


In [ ]:
import subprocess, gc, time
import numpy as np
import pandas as pd
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytabkit"], check=False)
from pytabkit import TabM_D_Classifier

DEV = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("[tabm] device", DEV)

# ── [추가] 파일 1과 완전히 동일한 v2v3 파생변수 생성기 (행단위 독립, 누수 0) ──
def NUM(df, c): return pd.to_numeric(df[c], errors="coerce") if c in df else pd.Series(np.nan, index=df.index)
def DIV(num, den): den = den.astype(float); return num.astype(float) / den.where(den > 0)

def masks(df):
    return {"신선": NUM(df, "신선 배아 사용 여부") == 1, "동결": NUM(df, "동결 배아 사용 여부") == 1,
            "ICSI": NUM(df, "미세주입된 난자 수") > 0,
            "본인난자": df["난자 출처"].astype(str) == "본인 제공", "기증난자": df["난자 출처"].astype(str) == "기증 제공"}

def build_v2_gated(df):
    Mk = masks(df); F = {}
    P1 = DIV(NUM(df, "총 생성 배아 수"), NUM(df, "혼합된 난자 수"))
    P2 = DIV(NUM(df, "미세주입에서 생성된 배아 수"), NUM(df, "미세주입된 난자 수"))
    P6 = DIV(NUM(df, "총 생성 배아 수"), NUM(df, "수집된 신선 난자 수"))
    L3 = NUM(df, "배아 이식 경과일") - NUM(df, "난자 혼합 경과일")
    F["g신선_수정률"] = P1.where(Mk["신선"])
    F["gICSI_수정효율"] = P2.where(Mk["ICSI"])
    F["g본인_난자수율"] = P6.where(Mk["본인난자"])
    F["g기증_난자수율"] = P6.where(Mk["기증난자"])
    F["g신선_배양일수"] = L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"] = (NUM(df, "배아 이식 경과일") - NUM(df, "배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"] = DIV(NUM(df, "이식된 배아 수"), NUM(df, "해동된 배아 수"))
    F["FZ3_해동난자수율"] = DIV(NUM(df, "총 생성 배아 수"), NUM(df, "해동 난자 수"))
    F["PG1_PGT강도"] = NUM(df, "착상 전 유전 검사 사용 여부").fillna(0) + NUM(df, "착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"] = NUM(df, "PGD 시술 여부").fillna(0) + NUM(df, "PGS 시술 여부").fillna(0)
    F["ST1_자극"] = NUM(df, "배란 자극 여부").fillna(0)
    return pd.DataFrame(F, index=df.index)

COL_RSN = "배아 생성 주요 이유"
def build_new_derived(df):
    F = {}
    tx = NUM(df, "이식된 배아 수").fillna(0); sto = NUM(df, "저장된 배아 수").fillna(0)
    ses = (df["단일 배아 이식 여부"] == 1).values
    F["EL_set_type"] = np.where(~ses, 0, np.where(sto.values > 0, 2, 1)).astype("int8")
    F["FA_no_transfer"] = (tx == 0).astype("int8").values
    is_current = df[COL_RSN].astype(str).str.contains("현재 시술용", na=False).values
    F["FA_nontransfer_reason"] = (~is_current).astype("int8")
    return pd.DataFrame(F, index=df.index)
# ───────────────────────────────────────────────────────────────────

def _deep_clf(seed):
    return TabM_D_Classifier(device=DEV, random_state=seed, n_cv=1, n_refit=0,
                             val_metric_name="cross_entropy", verbosity=0)

def _deep_base(df):
    D = df.drop(columns=[c for c in [TARGET, ID_COL] if c in df.columns]).copy()
    for c in OCC:
        if c in D: D[c] = D[c].astype(object).map(CMAP)
    for c, m in AGE_MAPS.items():
        if c in D: D[c] = D[c].astype(object).map(m)
        
    # ★ [핵심 수정] TabM도 트리/NN과 동일하게 v2+v3 핵심 파생변수를 결합하여 돌립니다.
    v2 = build_v2_gated(df)
    v3 = build_new_derived(df)
    D = pd.concat([D.reset_index(drop=True), v2.reset_index(drop=True), v3.reset_index(drop=True)], axis=1)
    return D

def _deep_prep(fit_df, dfs):
    F = _deep_base(fit_df)
    keep = [c for c in F.columns if F[c].nunique(dropna=True) > 1]
    objk = [c for c in keep if not pd.api.types.is_numeric_dtype(F[c])]
    numk = [c for c in keep if c not in objk]
    med = {}
    for c in numk:
        mm = pd.to_numeric(F[c], errors="coerce").median()
        med[c] = float(mm) if np.isfinite(mm) else 0.0
    outs = []
    for df in dfs:
        D = _deep_base(df).reindex(columns=keep)
        for c in numk: D[c] = pd.to_numeric(D[c], errors="coerce").fillna(med[c]).astype(np.float32)
        for c in objk: D[c] = D[c].astype(str).fillna("NA")
        outs.append(D.reset_index(drop=True))
    return outs, list(objk)

DETERMINISTIC = True
try: torch.use_deterministic_algorithms(True)
except Exception: DETERMINISTIC = False

def _fit(seed, X, yt, cat_names):
    global DETERMINISTIC
    clf = _deep_clf(seed)
    try:
        clf.fit(X, yt, cat_col_names=cat_names)
    except RuntimeError as e:
        if DETERMINISTIC and "deterministic" in str(e).lower():
            print("    ⚠️ 결정적 커널 미지원 → use_deterministic_algorithms(False) 폴백")
            DETERMINISTIC = False; torch.use_deterministic_algorithms(False)
            clf = _deep_clf(seed); clf.fit(X, yt, cat_col_names=cat_names)
        else: raise
    return clf

def tabm_member(seed=42):
    folds = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(train, y))
    oof = np.full(N, np.nan); tt = np.zeros(len(test)); t0 = time.perf_counter()
    for fi, (tri, vai) in enumerate(folds):
        (Xt, Xv, Xte), cat_names = _deep_prep(train.iloc[tri], [train.iloc[tri], train.iloc[vai], test])
        clf = _fit(seed, Xt, y[tri], cat_names)
        oof[vai] = clf.predict_proba(Xv)[:, 1]; tt += clf.predict_proba(Xte)[:, 1] / len(folds)
        print(f"    fold{fi} AUC={roc_auc_score(y[vai], oof[vai]):.5f} | 누적 {(time.perf_counter()-t0)/60:.1f}분", flush=True)
        del clf, Xt, Xv, Xte; gc.collect()
        try:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception: pass
    return oof, tt

oof_tabm, test_tabm = tabm_member(seed=42)
print(f"tabm 5-fold OOF AUC = {roc_auc_score(y, oof_tabm):.5f}  (det={DETERMINISTIC})")
assert roc_auc_score(y, oof_tabm) < 0.999, "라벨 누수 의심"
pd.DataFrame({"oof_tabm": oof_tabm, "y": y}).to_csv("oof_tabm.csv", index=False)
pd.DataFrame({"ID": test[ID_COL].values, "test_tabm": test_tabm}).to_csv("test_tabm.csv", index=False)
print("💾 oof_tabm.csv · test_tabm.csv 저장 — 파일3 입력")

[tabm] device cuda


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


    fold0 AUC=0.73758 | 누적 5.5분


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


    fold1 AUC=0.74270 | 누적 9.3분


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


    fold2 AUC=0.74117 | 누적 13.7분


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


    fold3 AUC=0.73818 | 누적 17.8분


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
